# Finding changes in Bradford's air quality with AQEval

This is the Python version of the [interactive tutorial](https://chris-r-uol.github.io/AQEval_Tutorial/).

**How to use it.** Click on a grey code box and press **Shift+Enter** to run it and move to the next. Run the boxes in order, from the top. The first time, choose the **Python 3** kernel if you are asked.

## 1. Load the packages

aqeval is the Python version of the R package AQEval. It downloads UK air quality data and finds the changes in it.

In [ ]:
import aqeval

## 2. Get the data

Download every hourly measurement from 2018 to 2025 for one monitoring site on the national network (AURN). Each site has a short code. The download also brings the wind speed, wind direction and air temperature for each hour.

In [ ]:
data = aqeval.download_aurn_data(
    "bdma", 2018, 2025, source="aurn"
)

## 3. Look at the data

Always plot the measurements before analysing them. Look for gaps, and for anything that seems out of place.

In [ ]:
data.plot(x="date", y="no2")

## 4. Isolate the signal

Pollution rises and falls with the weather, the time of day and the time of year. This step fits a model of those patterns and takes them away, leaving the part of the signal that a change in emissions could explain.

In [ ]:
data["isolated"] = aqeval.isolate_contribution(
    data, "no2",
    background="air_temp",
    deseason=True,
    deweather=True,
)

## 5. Average the data

Hourly values are noisy. Averaging them into longer blocks smooths out the short-lived ups and downs, and gives the next step fewer points to work through.

In [ ]:
data_avg = aqeval.time_average(data, "8 hour")

## 6. Find the break points

A break point is a moment where the average level of the series shifts. h sets the shortest stretch allowed between two breaks, as a fraction of the whole series: a smaller h can find more breaks, closer together, and takes longer. 0.3 is a quick first look. For 8-hour data we would normally use 0.12.

In [ ]:
breaks = aqeval.find_break_points(
    data_avg, "isolated", h=0.3
)
breaks

## 7. Measure the changes

Real changes rarely happen in a single day. This step fits a line through each stretch of the series and reports when each change started, when it finished, and how big it was.

In [ ]:
result = aqeval.quant_break_segments(
    data_avg, "isolated", breaks=breaks
)
result["report"]

## Things to try

1. Change "no2" to "nox". Do the changes fall on the same dates?
2. Change the site from "bdma" to "led6" (Leeds Headingley Kerbside), a roadside site with no Clean Air Zone.
3. Make the analysis more sensitive: change h from 0.3 to 0.2, then to 0.12, the value normally used for 8-hour data. Smaller values find more changes. On 8-hour data, measuring the changes then takes several minutes, so change "8 hour" to "day" first: on daily data it takes under a minute.
4. Use a background site as a control: the website shows the extra lines when you choose one under "Also account for".